# From raw reads to drug-resistance prediction in *Mycobacterium tuberculosis*

**Question.** Starting from raw Illumina reads, can a standard variant-calling pipeline find the mutations that make TB
bacteria resistant to the two key first-line drugs, **rifampicin (RIF)** and **isoniazid (INH)**, and do its
predictions agree with resistance measured in the laboratory?

**Data.** Clinical isolates from the CRyPTIC consortium (whole-genome sequencing plus laboratory-measured drug
susceptibility; tables on Zenodo, reads in the European Nucleotide Archive). Reference genome: H37Rv
(NC_000962.3, NCBI RefSeq).

**Pipeline.** fastp (trimming, QC) → BWA-MEM (alignment) → samtools (sorting, coverage) → bcftools (variant calling,
haploid) → annotation in Python → rule-based prediction → comparison with the lab results.

**How to run.** `Runtime → Run all`. No sign-in is needed: all data are open. The whole notebook takes about
45–60 minutes, most of it in Step 5. At the end `results.zip` downloads to your computer.

## Step 1. Install the bioinformatics tools
These are the standard command-line tools of NGS analysis. `!` at the start of a line runs it in the Linux shell
of the Colab machine.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y bwa samtools bcftools fastp tabix > /dev/null
!bwa 2>&1 | grep Version; samtools --version | head -1; bcftools --version | head -1; fastp --version

In [ ]:
import os, io, gzip, json, subprocess, textwrap, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
for d in ("ref", "reads", "bam", "vcf", "results"):
    os.makedirs(d, exist_ok=True)

def sh(cmd, check=True):
    # Run a shell command, show it, and stop with its error message if it fails.
    print("$", cmd if len(cmd) < 160 else cmd[:157] + "...")
    p = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(p.stderr[-2000:])
    return p.stdout

## Step 2. Helper functions: annotation and resistance rules
Read this cell carefully: it is the biology of the project.
- `annotate` works out what a variant does to the protein, e.g. `rpoB_S450L` = serine at codon 450 of rpoB replaced
  by leucine. For genes on the minus strand it uses the reverse complement.
- `predict` applies a **simplified** set of high-confidence rules from the WHO mutation catalogue:
  - **RIF resistant** if any amino-acid change in the rifampicin-resistance-determining region of *rpoB* (codons 426–452);
  - **INH resistant** if *katG* codon 315 is changed, *katG* is truncated (stop codon or frameshift), or the *fabG1*
    promoter carries c.-15C>T or c.-8T>C/A (this promoter drives *inhA*, the isoniazid target).

In [ ]:
# ---- core functions shared by the notebook (tested locally on synthetic data) ----
import gzip, re
import numpy as np
import pandas as pd

COMP = str.maketrans("ACGTacgtN", "TGCAtgcaN")
def revcomp(s): return s.translate(COMP)[::-1]

BASES = "TCAG"
AA = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
CODON = {a + b + c: AA[16 * i + 4 * j + k]
         for i, a in enumerate(BASES) for j, b in enumerate(BASES) for k, c in enumerate(BASES)}
def translate(codon): return CODON.get(codon.upper(), "X")

def read_fasta(path):
    seq = []
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as f:
        for line in f:
            if not line.startswith(">"):
                seq.append(line.strip())
    return "".join(seq).upper()

def read_genes(gff_path, wanted):
    """wanted: {display_name: [aliases]} -> {display_name: (start, end, strand)} from a GFF3 'gene' feature."""
    found = {}
    opener = gzip.open if str(gff_path).endswith(".gz") else open
    with opener(gff_path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            p = line.rstrip("\n").split("\t")
            if len(p) < 9 or p[2] != "gene":
                continue
            attrs = dict(kv.split("=", 1) for kv in p[8].split(";") if "=" in kv)
            names = {attrs.get(k) for k in ("Name", "gene", "locus_tag", "old_locus_tag")}
            for disp, aliases in wanted.items():
                if disp not in found and names & set(aliases):
                    found[disp] = (int(p[3]), int(p[4]), p[6])
    return found

def annotate(pos, ref, alt, genes, genome, upstream=200):
    """Describe a variant relative to the genes of interest. Returns (gene, label, kind, codon_number)."""
    for g, (start, end, strand) in genes.items():
        snp = len(ref) == 1 and len(alt) == 1
        if start <= pos <= end:
            if not snp:
                kind = "frameshift" if (len(alt) - len(ref)) % 3 else "inframe_indel"
                i = (pos - start) if strand == "+" else (end - pos)
                return g, f"{g}_indel_codon{i // 3 + 1}", kind, i // 3 + 1
            if strand == "+":
                i = pos - start
                cds = genome[start - 1:end]
                a = alt
            else:
                i = end - pos
                cds = revcomp(genome[start - 1:end])
                a = revcomp(alt)
            k, off = i // 3, i % 3
            codon = cds[3 * k:3 * k + 3]
            mut = codon[:off] + a + codon[off + 1:]
            r_aa, a_aa = translate(codon), translate(mut)
            kind = "synonymous" if r_aa == a_aa else ("stop_gained" if a_aa == "*" else "missense")
            return g, f"{g}_{r_aa}{k + 1}{a_aa}", kind, k + 1
        # promoter / upstream
        if snp and strand == "+" and start - upstream <= pos < start:
            return g, f"{g}_c.{pos - start}{ref}>{alt}", "upstream", None
        if snp and strand == "-" and end < pos <= end + upstream:
            return g, f"{g}_c.-{pos - end}{revcomp(ref)}>{revcomp(alt)}", "upstream", None
    return None, None, None, None

def codon_at(genes, genome, gene, n):
    start, end, strand = genes[gene]
    cds = genome[start - 1:end] if strand == "+" else revcomp(genome[start - 1:end])
    return translate(cds[3 * (n - 1):3 * n])

def predict(row_mutations):
    """Simplified rule set based on high-confidence WHO-catalogue mutations. Returns dict drug -> 'R'/'S' and reasons."""
    rif, inh = [], []
    for m in row_mutations:
        g, label, kind, codon = m["gene"], m["label"], m["kind"], m["codon"]
        if g == "rpoB" and kind in ("missense", "inframe_indel") and codon and 426 <= codon <= 452:
            rif.append(label)
        if g == "katG" and kind == "missense" and codon == 315:
            inh.append(label)
        if g == "katG" and kind in ("stop_gained", "frameshift"):
            inh.append(label)
        if g == "fabG1" and label in ("fabG1_c.-15C>T", "fabG1_c.-8T>C", "fabG1_c.-8T>A"):
            inh.append(label)
    return {"RIF": ("R" if rif else "S", ";".join(rif)), "INH": ("R" if inh else "S", ";".join(inh))}

# ---- finding the right columns in tables whose layout we cannot see in advance ----
RUN_RE = re.compile(r"^[EDS]RR\d+")

def text_columns(df):
    # Parquet files often store text as "category" columns, so those count as text too.
    return [c for c in df.columns if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
            or isinstance(df[c].dtype, pd.CategoricalDtype)]

PHENO_MAP = {"R": "R", "S": "S", "RESISTANT": "R", "SUSCEPTIBLE": "S", "SENSITIVE": "S", "U": "U", "I": "I"}

def norm_values(series):
    return series.dropna().astype(str).str.strip().str.upper()

def norm_drug(series):
    v = series.astype(str).str.strip().str.upper()
    return v.replace({"RIFAMPICIN": "RIF", "RIFAMPIN": "RIF", "ISONIAZID": "INH"})

def share_matching(series, pattern):
    s = series.dropna().astype(str).head(2000)
    return 0 if len(s) == 0 else s.str.match(pattern).mean()

def detect_run_column(df):
    scores = {c: share_matching(df[c], RUN_RE) for c in text_columns(df)}
    best = max(scores, key=scores.get) if scores else None
    return best if best and scores[best] > 0.5 else None

def detect_phenotype_layout(df):
    """Long layout: a drug column (values include RIF and INH) + a phenotype column (values R/S).
    Wide layout: columns like RIF_..._PHENOTYPE holding R/S. Returns a long DataFrame [id_col?, DRUG, PHENOTYPE, QUALITY]."""
    obj = text_columns(df)
    drug_col = next((c for c in obj if {"RIF", "INH"} <= set(norm_drug(df[c].dropna()).unique())), None)
    def rs(c):
        vals = set(norm_values(df[c]).map(PHENO_MAP).unique())
        return {"R", "S"} <= vals and not (set(norm_values(df[c]).unique()) - set(PHENO_MAP))
    if drug_col:
        phen_col = next((c for c in obj if c != drug_col and rs(c)), None)
        qual_col = next((c for c in df.columns if "QUALITY" in c.upper()), None)
        if phen_col:
            return {"layout": "long", "drug": drug_col, "phenotype": phen_col, "quality": qual_col}
    wide = {d: next((c for c in obj if c.upper().startswith(d) and rs(c)), None) for d in ("RIF", "INH")}
    if all(wide.values()):
        return {"layout": "wide", **wide}
    return None

## Step 3. Reference genome and genes of interest
The checks at the end confirm the gene coordinates are right: known codons (e.g. serine at *rpoB* 450, the site of
the most common rifampicin-resistance mutation) must read correctly. If any check fails, stop and report it.

In [ ]:
BASE = "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/195/955/GCF_000195955.2_ASM19595v2/GCF_000195955.2_ASM19595v2"
sh(f"wget -q -O ref/h37rv.fna.gz {BASE}_genomic.fna.gz && wget -q -O ref/h37rv.gff.gz {BASE}_genomic.gff.gz")
sh("gunzip -f ref/h37rv.fna.gz && samtools faidx ref/h37rv.fna && bwa index ref/h37rv.fna 2> /dev/null")
CHROM = open("ref/h37rv.fna").readline()[1:].split()[0]
genome = read_fasta("ref/h37rv.fna")
print(CHROM, len(genome), "bp")

WANTED = {"rpoB": ["rpoB", "Rv0667"], "katG": ["katG", "Rv1908c"], "fabG1": ["fabG1", "mabA", "Rv1483"],
          "inhA": ["inhA", "Rv1484"], "gyrA": ["gyrA", "Rv0006"], "embB": ["embB", "Rv3795"],
          "rpsL": ["rpsL", "Rv0682"], "pncA": ["pncA", "Rv2043c"]}
genes = read_genes("ref/h37rv.gff.gz", WANTED)
print(pd.DataFrame(genes, index=["start", "end", "strand"]).T)

checks = [("rpoB", 450, "S"), ("rpoB", 445, "H"), ("rpoB", 435, "D"), ("katG", 315, "S"),
          ("gyrA", 94, "D"), ("gyrA", 90, "A"), ("embB", 306, "M"), ("rpsL", 43, "K"), ("rpsL", 88, "K")]
for g, n, aa in checks:
    got = codon_at(genes, genome, g, n)
    print(f"{g} codon {n}: expected {aa}, found {got}", "OK" if got == aa else "<-- MISMATCH")
    assert got == aa, "Gene coordinates do not match expectations: stop and report this."
fab_start = genes["fabG1"][0]
assert genome[fab_start - 15 - 1] == "C", "fabG1 c.-15 is not C in the reference: stop and report this."
print("fabG1 c.-15 reference base: C  OK")

## Step 4. Choose the samples
CRyPTIC publishes its tables as Parquet files. This step downloads the two small ones (lab phenotypes and the list of
sequenced genomes), finds the right columns, and picks samples **at random** within three groups so the choice is
not cherry-picked:
- 6 multidrug-resistant (RIF-R and INH-R), 2 isoniazid-only resistant (RIF-S, INH-R), 4 susceptible to both.

Only samples with paired-end Illumina reads in a single sequencing run are used. If a phenotype-quality column exists,
only high-quality phenotypes are kept.

In [ ]:
ZEN = "https://zenodo.org/records/15680920/files"   # CRyPTIC Consortium Dataset v3.4.0 (fixed version, for reproducibility)
for f in ("UKMYC_PHENOTYPES.parquet", "WGS_SAMPLES.parquet"):
    if not os.path.exists(f"ref/{f}"):
        sh(f"wget -q -O ref/{f} '{ZEN}/{f}?download=1'")
ph = pd.read_parquet("ref/UKMYC_PHENOTYPES.parquet")
wgs = pd.read_parquet("ref/WGS_SAMPLES.parquet")
for name, df in (("UKMYC_PHENOTYPES", ph), ("WGS_SAMPLES", wgs)):
    print(f"\n{name}: {df.shape[0]} rows")
    print(df.head(3).to_string())

In [ ]:
# Parquet tables can keep their ID columns (e.g. UNIQUEID, DRUG) in the index; turn them back into columns first.
if isinstance(ph.index, pd.MultiIndex) or ph.index.name:
    ph = ph.reset_index()
if isinstance(wgs.index, pd.MultiIndex) or wgs.index.name:
    wgs = wgs.reset_index()
layout = detect_phenotype_layout(ph)
run_col = detect_run_column(wgs)
key = next((c for c in ph.columns if c in wgs.columns and "UNIQUEID" in c.upper()), None) or \
      next((c for c in ph.columns if c in wgs.columns), None)
print("phenotype layout:", layout, "| run column:", run_col, "| join key:", key)
if not (layout and run_col and key):
    print("\nDiagnostics (send a screenshot of this):")
    print("  index:", ph.index.names)
    for c in ph.columns:
        print(f"  {c!r:32} {str(ph[c].dtype):10} e.g. {list(ph[c].dropna().astype(str).unique()[:6])}")
    raise SystemExit("Could not find the columns automatically.")

if layout["layout"] == "long":
    p = ph.copy()
    if layout["quality"] and (p[layout["quality"]].astype(str).str.upper() == "HIGH").any():
        p = p[p[layout["quality"]].astype(str).str.upper() == "HIGH"]
    p = p.assign(_d=norm_drug(p[layout["drug"]]))
    p = p[p["_d"].isin(["RIF", "INH"])]
    p = p.assign(
                 _p=p[layout["phenotype"]].astype(str).str.strip().str.upper().map(PHENO_MAP))
    pheno = p.pivot_table(index=key, columns="_d", values="_p", aggfunc="first")
else:
    pheno = ph.set_index(key)[[layout["RIF"], layout["INH"]]].rename(columns={layout["RIF"]: "RIF", layout["INH"]: "INH"})
    pheno = pheno.apply(lambda s: s.astype(str).str.strip().str.upper().map(PHENO_MAP))
pheno = pheno[pheno["RIF"].isin(["R", "S"]) & pheno["INH"].isin(["R", "S"])]
runs = wgs[[key, run_col]].dropna().drop_duplicates(key).set_index(key)[run_col].astype(str)
runs = runs[~runs.str.contains(r"[,;\s]")]           # single sequencing run only
cand = pheno.join(runs.rename("run"), how="inner")
print(len(cand), "samples with RIF and INH phenotypes and a single sequencing run")
print(pd.crosstab(cand["RIF"], cand["INH"]))

In [ ]:
def ena_info(run):
    url = ("https://www.ebi.ac.uk/ena/portal/api/filereport?accession=" + run +
           "&result=read_run&fields=run_accession,fastq_ftp,library_layout,instrument_platform,read_count&format=tsv")
    try:
        t = pd.read_csv(url, sep="\t")
        return t.iloc[0].to_dict() if len(t) else None
    except Exception:
        return None

GROUPS = [("MDR", "R", "R", 6), ("INH-mono", "S", "R", 2), ("susceptible", "S", "S", 4)]
chosen = []
for label, rif, inh, n in GROUPS:
    pool = cand[(cand.RIF == rif) & (cand.INH == inh)].sample(frac=1, random_state=42)
    got = 0
    for sid, r in pool.iterrows():
        info = ena_info(r["run"])
        if not info or str(info.get("library_layout")) != "PAIRED" or str(info.get("instrument_platform")) != "ILLUMINA":
            continue
        urls = str(info.get("fastq_ftp", "")).split(";")
        if len(urls) != 2:
            continue
        chosen.append({"sample": str(sid), "run": r["run"], "group": label, "RIF_lab": rif, "INH_lab": inh,
                       "read_pairs_total": info.get("read_count"), "url1": "https://" + urls[0], "url2": "https://" + urls[1]})
        got += 1
        if got == n:
            break
    print(f"{label}: {got} of {n} samples selected")
samples = pd.DataFrame(chosen)
samples.drop(columns=["url1", "url2"]).to_csv("results/samples.csv", index=False)
samples.drop(columns=["url1", "url2"])

## Step 5. Run the pipeline on each sample (the slow step)
For each sample:
1. **Download** the first 750,000 read pairs only (streamed, so the full file is never downloaded). For a 4.4 Mb
   genome that is roughly 50× coverage, plenty for confident calls.
2. **fastp**: trim adapters and low-quality bases; writes a QC report.
3. **BWA-MEM**: align reads to H37Rv. **samtools**: sort and index the alignment (BAM file).
4. **samtools coverage**: how much of the genome and of each target gene is covered, and how deeply.
5. **bcftools**: call variants in the target genes with `--ploidy 1`, because bacteria are haploid.

In [ ]:
N_PAIRS = 750_000
REGIONS = ",".join(f"{CHROM}:{max(1, s - 200)}-{e + 200}" for s, e, _ in genes.values())
qc_rows = []
for i, r in samples.iterrows():
    s = r["sample"].replace("/", "_")
    print(f"\n=== {i + 1}/{len(samples)}  {s}  ({r['group']}) ===")
    for k in (1, 2):
        out = f"reads/{s}_R{k}.fq.gz"
        if not os.path.exists(out):
            sh(f"curl -sSL --retry 3 '{r[f'url{k}']}' | gzip -dc 2>/dev/null | head -n {4 * N_PAIRS} | gzip -1 > {out}", check=False)
    sh(f"fastp -i reads/{s}_R1.fq.gz -I reads/{s}_R2.fq.gz -o reads/{s}_t1.fq.gz -O reads/{s}_t2.fq.gz "
       f"-w 2 -j results/{s}_fastp.json -h /dev/null 2> /dev/null")
    sh(f"bwa mem -t 2 -R '@RG\\tID:{s}\\tSM:{s}' ref/h37rv.fna reads/{s}_t1.fq.gz reads/{s}_t2.fq.gz 2> /dev/null "
       f"| samtools sort -@2 -o bam/{s}.bam - && samtools index bam/{s}.bam")
    cov = pd.read_csv(io.StringIO(sh(f"samtools coverage bam/{s}.bam")), sep="\t")
    row = {"sample": r["sample"], "genome_covered_pct": cov["coverage"].iloc[0], "mean_depth": cov["meandepth"].iloc[0]}
    for g, (gs, ge, _) in genes.items():
        c = pd.read_csv(io.StringIO(sh(f"samtools coverage -r {CHROM}:{gs}-{ge} bam/{s}.bam")), sep="\t")
        row[f"depth_{g}"] = c["meandepth"].iloc[0]
    fp = json.load(open(f"results/{s}_fastp.json"))
    row["reads_after_trim"] = fp["summary"]["after_filtering"]["total_reads"]
    row["q30_rate"] = fp["summary"]["after_filtering"]["q30_rate"]
    qc_rows.append(row)
    sh(f"bcftools mpileup -f ref/h37rv.fna -r {REGIONS} -q 20 -Q 20 -a FORMAT/AD,FORMAT/DP -Ou bam/{s}.bam 2> /dev/null "
       f"| bcftools call -m -v --ploidy 1 -Oz -o vcf/{s}.vcf.gz 2> /dev/null && bcftools index -f vcf/{s}.vcf.gz")
    for f in (f"reads/{s}_t1.fq.gz", f"reads/{s}_t2.fq.gz"):
        os.remove(f)
qc = pd.DataFrame(qc_rows)
qc.to_csv("results/qc.csv", index=False)
qc.round(2)

## Step 6. Annotate the variants
Each variant is kept if it passes basic quality filters (QUAL ≥ 20, depth ≥ 10) and is described relative to the
genes of interest. `alt_fraction` is the share of reads carrying the mutation: close to 1 for a clean haploid call;
clearly below 1 can mean a mixed infection or heteroresistance.

Note: many synonymous and some missense variants seen here are **lineage markers** (they distinguish TB lineages)
and have nothing to do with resistance. That is why a curated catalogue is needed, not "any mutation = resistant".

In [ ]:
def read_vcf(path):
    rows = []
    with gzip.open(path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            c = line.rstrip("\n").split("\t")
            info = dict(kv.split("=", 1) for kv in c[7].split(";") if "=" in kv)
            fmt = dict(zip(c[8].split(":"), c[9].split(":")))
            ad = [int(x) for x in fmt.get("AD", "0").split(",") if x not in (".", "")]
            depth = sum(ad) if ad else int(info.get("DP", 0))
            for j, alt in enumerate(c[4].split(",")):
                rows.append({"pos": int(c[1]), "ref": c[3], "alt": alt, "qual": float(c[5]),
                             "depth": depth, "alt_fraction": (ad[j + 1] / depth) if len(ad) > j + 1 and depth else np.nan})
    return pd.DataFrame(rows)

var_rows = []
for s in samples["sample"]:
    v = read_vcf(f"vcf/{s.replace('/', '_')}.vcf.gz")
    if v.empty:
        continue
    v = v[(v.qual >= 20) & (v.depth >= 10)]
    for _, x in v.iterrows():
        g, label, kind, codon = annotate(x.pos, x.ref, x.alt, genes, genome)
        if g:
            var_rows.append({"sample": s, **x.to_dict(), "gene": g, "label": label, "kind": kind, "codon": codon})
variants = pd.DataFrame(var_rows)
variants.to_csv("results/variants_annotated.csv", index=False)
print(len(variants), "variants in the genes of interest")
variants[variants.kind != "synonymous"].sort_values(["gene", "pos"]).head(40)

## Step 7. Predict resistance and compare with the laboratory
- **Sensitivity**: of the isolates the lab found resistant, the share the pipeline also called resistant.
- **Specificity**: of the isolates the lab found susceptible, the share the pipeline called susceptible.

With only 12 samples the 95 % confidence intervals are wide. This is a demonstration of the pipeline, not an
evaluation of diagnostic accuracy.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (c - h, c + h)

pred_rows = []
for _, r in samples.iterrows():
    muts = variants[variants["sample"] == r["sample"]].to_dict("records") if len(variants) else []
    p = predict(muts)
    pred_rows.append({"sample": r["sample"], "group": r["group"],
                      "RIF_lab": r["RIF_lab"], "RIF_pred": p["RIF"][0], "RIF_mutations": p["RIF"][1],
                      "INH_lab": r["INH_lab"], "INH_pred": p["INH"][0], "INH_mutations": p["INH"][1]})
pred = pd.DataFrame(pred_rows)
pred.to_csv("results/predictions.csv", index=False)

perf = []
for drug in ("RIF", "INH"):
    lab, pr = pred[f"{drug}_lab"], pred[f"{drug}_pred"]
    tp, fn = int(((lab == "R") & (pr == "R")).sum()), int(((lab == "R") & (pr == "S")).sum())
    tn, fp = int(((lab == "S") & (pr == "S")).sum()), int(((lab == "S") & (pr == "R")).sum())
    se, sp = wilson(tp, tp + fn), wilson(tn, tn + fp)
    perf.append({"drug": drug, "TP": tp, "FN": fn, "TN": tn, "FP": fp,
                 "sensitivity": tp / (tp + fn) if tp + fn else np.nan, "sens_lo": se[0], "sens_hi": se[1],
                 "specificity": tn / (tn + fp) if tn + fp else np.nan, "spec_lo": sp[0], "spec_hi": sp[1]})
perf = pd.DataFrame(perf)
perf.to_csv("results/performance.csv", index=False)
display(pred)
perf.round(2)

### Discordant samples
Any sample where the pipeline and the lab disagree is the most interesting result. Look at its mutations, its
coverage of the relevant gene, and its `alt_fraction`, and think about why.

In [ ]:
for drug in ("RIF", "INH"):
    bad = pred[pred[f"{drug}_lab"] != pred[f"{drug}_pred"]]
    print(f"\n{drug}: {len(bad)} discordant sample(s)")
    for _, r in bad.iterrows():
        s = r["sample"]
        q = qc[qc["sample"] == s].iloc[0]
        g = "rpoB" if drug == "RIF" else "katG"
        print(f"  {s}: lab {r[f'{drug}_lab']}, predicted {r[f'{drug}_pred']}; depth in {g}: {q[f'depth_{g}']:.0f}x")
        v = variants[(variants["sample"] == s) & (variants.kind != "synonymous")]
        print(textwrap.indent(v[["label", "kind", "alt_fraction", "depth"]].to_string(index=False), "    ") if len(v) else "    no non-synonymous variants in the genes of interest")

## Step 8. Figure: resistance mutations by sample
Rows are samples (grouped by lab result); columns are the mutations the rules use, plus any other non-synonymous
change seen in the resistance genes. The two left columns show the lab result.

In [ ]:
used = set(";".join(pred["RIF_mutations"].tolist() + pred["INH_mutations"].tolist()).split(";")) - {""}
other = set(variants.loc[variants.kind.isin(["missense", "stop_gained", "frameshift", "inframe_indel", "upstream"]), "label"]) if len(variants) else set()
cols = sorted(used) + sorted(other - used)
order = pred.sort_values(["group", "sample"])
mat = np.zeros((len(order), len(cols)))
for i, s in enumerate(order["sample"]):
    vs = variants[variants["sample"] == s] if len(variants) else pd.DataFrame(columns=["label", "alt_fraction"])
    for j, c in enumerate(cols):
        hit = vs[vs["label"] == c]
        if len(hit):
            mat[i, j] = hit["alt_fraction"].fillna(1).max()

fig, ax = plt.subplots(figsize=(max(7, 2.6 + 0.45 * len(cols)), 1.6 + 0.38 * len(order)))
ax.imshow(mat, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(order)), [f"{s[:18]}  RIF {a} / INH {b}" for s, a, b in zip(order["sample"], order["RIF_lab"], order["INH_lab"])], fontsize=8)
if used:
    ax.axvline(len(used) - 0.5, color="#52514e", lw=1)
ax.set_title("Resistance-gene mutations per sample\n(shade = fraction of reads; left of the line = used by the rules)", loc="left", fontsize=9)
fig.tight_layout()
fig.savefig("results/mutation_matrix.png", dpi=200)
plt.show()

## Step 9. Save everything

In [ ]:
import shutil
shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("results.zip is in the current folder")

## Questions to answer in your own words (interview preparation)
1. Why is `--ploidy 1` used, and what would go wrong with the default (diploid) setting?
2. What does BWA-MEM produce, and what is in a BAM file? What is a VCF file?
3. Why stream only 750,000 read pairs? How would you decide how many reads are "enough"?
4. What is the RRDR of *rpoB*, and why do mutations there cause rifampicin resistance?
5. *katG* encodes an enzyme that *activates* isoniazid. Why does losing katG cause resistance, while mutations in the
   *inhA* promoter cause it by a different route?
6. Look at every discordant sample. Give at least two possible explanations for each.
7. Many variants are lineage markers. How would you tell a lineage marker from a resistance mutation?
8. What would you change to evaluate accuracy properly rather than demonstrate the pipeline?